# Cross-model tournament — starting point

Runs trained Octagon models against each other in inference and writes one behavioural-log JSON per matchup (the format `octagon_analysis` consumes). See `tournament/tournament_func.py` and `tournament/Tournament_plan.md`.

**Before running:**
1. Activate the conda env that has `mlagents` (the kernel for this notebook must be that env).
2. Make a **fresh** `TournamentOctagonStage` build (headless Linux `.x86_64`). A fresh build is required so it has `ScriptedTrialSequence.cs` + the timing keys — an older build silently ignores `--trial_seq` and generates random trials. Set its path in `UNITY_ENV_PATH` below.
3. Entrants must be `OctagonAgentSocial` models from the `260715` build or newer (they match the tournament build's 110° FoV observations).

Actions are **sampled** (not greedy), matching training behaviour → matchups are not bit-reproducible; run enough episodes to average over sampling noise. Trial content/timing is pinned by the predetermined `trial_seq`.

In [1]:
import sys
from pathlib import Path

# Repo root on sys.path so `tournament.tournament_func` and the
# `trainer_and_simulator_functions` it imports both resolve.
REPO_ROOT = Path("~/repos/agent_training").expanduser()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tournament.tournament_func import run_tournament, run_matchup, write_tournament_config_yaml

## Config

Edit the paths / picks here. `MODELS_ROOT` is the parent folder; entrants are its immediate subfolders (each with `OctagonAgentSocial/checkpoint.pt`). All entrants must live under one `MODELS_ROOT` — to mix `260715` and `260723`, gather the chosen run dirs under a single folder first.

In [ ]:
OCTAGON_DIR = Path("~/Unity/Octagon").expanduser()

# Fresh TournamentOctagonStage build (headless Linux).
UNITY_ENV_PATH = Path("~/Unity/Octagon/builds/260724/build_260724_01_tournament.x86_64").expanduser()

# Parent folder holding the entrant run dirs (each an immediate subfolder).
MODELS_ROOT = Path("~/Unity/Octagon/results/2607/260715").expanduser()

# Hand-picked entrants (subfolder names under MODELS_ROOT), per tournament_notes_260715.txt.
# Set to None to auto-use EVERY model under MODELS_ROOT (careful: N models => C(N,2)+N matchups).
RUN_IDS = [
    "260715_0022",  # most central (top scorer, player score)
    "260715_0020",  # strong central (2nd scorer)
    "260715_0002",  # 'typical' centre
    "260715_0014",  # intermediate (central leaning)
    "260715_0010",  # intermediate (peripheral leaning) - slower completion tail
    "260715_0036",  # best peripheral scorer + prompt (replaces old 0005)
    "260715_0015",  # peripheral, most High-seeking
    "260715_0026",  # 'typical' peripheral (3e6)
]


# Predetermined trial sequence (pins trial content + timing across all matchups).
# Keep it LONGER than EPISODES — the build stops when the sequence is exhausted.
TRIAL_SEQ = Path("~/repos/agent_training/trial_sequences/trials_100000_seed17.json").expanduser()

# Where the per-matchup JSON logs land. Prior runs are archived under
# simulations/tournaments/archived/<build_date>/.
OUT_ROOT = Path("~/Unity/Octagon/simulations/tournaments").expanduser()

# MLAgents episodes per matchup. Conversion (measured on 260715 inference, episodes=201
# -> ~200 "trial start" events): trials ~= episodes - 1, i.e. ~1 trial per episode.
# So for a target of T trials/matchup, set EPISODES ~= T + 1 (occasional runs finish a
# trial or two short under sampled actions, so treat it as an upper bound).
EPISODES = 350   # ~350 trials/matchup (T+1 per the conversion above)

# Run each distinct pair in BOTH seat orders (mirror matchups). P1/P2 is symmetric
# so this is off in the function by default; ON here to double data (2 x ~250 = ~500
# trials/pair) and to confirm the seat has no effect by comparing a pair vs its mirror.
INCLUDE_MIRRORS = True

In [3]:
# Fail fast with a clear message if anything is missing.
assert UNITY_ENV_PATH.exists(), f"Build not found: {UNITY_ENV_PATH} (set UNITY_ENV_PATH to your fresh tournament build)"
assert MODELS_ROOT.exists(), f"MODELS_ROOT not found: {MODELS_ROOT}"
assert TRIAL_SEQ.exists(), f"TRIAL_SEQ not found: {TRIAL_SEQ}"
for rid in (RUN_IDS or []):
    ckpts = list((MODELS_ROOT / rid).glob("*/checkpoint.pt"))
    assert ckpts, f"No <behaviour>/checkpoint.pt under {MODELS_ROOT / rid}"
n = len(RUN_IDS) if RUN_IDS else len(list(MODELS_ROOT.glob('*/*/checkpoint.pt')))
distinct_pairs = n * (n - 1) // 2
n_matchups = distinct_pairs * (2 if INCLUDE_MIRRORS else 1) + n   # + n self-matchups
print(f"All paths OK. {n} entrants => {n_matchups} matchups "
      f"({'with' if INCLUDE_MIRRORS else 'no'} mirrors, incl. self-matchups).")

All paths OK. 8 entrants => 64 matchups (with mirrors, incl. self-matchups).


## (Optional) Single-matchup smoke test

Run one matchup with a few episodes first to confirm the build + checkpoints load and a JSON is produced, before committing to the full round-robin. `run_matchup` needs the CLI config yaml to already exist (the full `run_tournament` writes it for you), so we write it here first.

In [ ]:
config_yaml = write_tournament_config_yaml(
    template_run_dir=MODELS_ROOT / RUN_IDS[0],
    out_yaml=OUT_ROOT / "tournament_config.yaml",
)

log = run_matchup(
    model_a_run_dir=MODELS_ROOT / RUN_IDS[0],
    model_b_run_dir=MODELS_ROOT / RUN_IDS[1],
    octagon_dir=OCTAGON_DIR,
    unity_env_path=UNITY_ENV_PATH,
    out_path=OUT_ROOT / "_smoke_test",
    episodes=5,
    config_yaml=config_yaml,
    trial_seq=TRIAL_SEQ,
)
print("smoke-test log:", log)

## Full round-robin

`combinations_with_replacement` — every unordered pair, **including self-matchups**. Mirrors (seat-swapped duplicates) are **optional** via `include_mirrors` (default off in the function; set by `INCLUDE_MIRRORS` above). One JSON per matchup under `OUT_ROOT/<A>__vs__<B>/`. Failures are caught per-matchup (that entry is `None`) so one bad run doesn't kill the batch.

**Self-matchups vs mirrors — to be clear:**
- **Self-matchups (a model vs a copy of itself) ARE run** — one per entrant, giving a ~50% baseline / sanity check.
- **Mirrors (the same pairing with the P1/P2 seats swapped) are OFF by default.** Which seat a model takes is irrelevant: agents aren't repositioned at trial start, so each agent's position is emergent from its own behaviour, and the only P1/P2 difference is administrative (arena setup / logging). Set `INCLUDE_MIRRORS = True` to run both seat orders — this doubles data per pair and lets you confirm empirically that the seat has no effect (compare A-vs-B against B-vs-A on the same trial sequence).

In [4]:
logs = run_tournament(
    models_root=MODELS_ROOT,
    octagon_dir=OCTAGON_DIR,
    unity_env_path=UNITY_ENV_PATH,
    out_root=OUT_ROOT,
    episodes=EPISODES,
    run_ids=RUN_IDS,            # None => all models under MODELS_ROOT
    include_self_matchups=True,
    include_mirrors=INCLUDE_MIRRORS,
    trial_seq=TRIAL_SEQ,
    base_port=9100,             # ports = 9100 + 20*i -> 9100..10360. Training uses
                                # 5005+20*i per model, so a session tops ~6400 (70 models)
                                # / ~9000 (200 models); 9100 clears sessions up to ~206
                                # models. Also above tensorboard (6006) / mDNS (5353).
)


Running 64 matchups over 8 models: ['260715_0022', '260715_0005', '260715_0002', '260715_0026', '260715_0024', '260715_0010', '260715_0015', '260715_0014']

=== [1/64] Matchup: 260715_0022 (P1) vs 260715_0022 (P2) ===
['mlagents-learn', '/home/tom/repos/agent_training/tournament/results/tournament_config.yaml', '--run-id', 'tournament/260715_0022__vs__260715_0022', '--resume', '--inference', '--base-port', '9100', '--env', '/home/tom/Unity/Octagon/builds/260724/build_260724_01_tournament.x86_64', '--no-graphics', '--env-args', '--sim_out', '/home/tom/repos/agent_training/tournament/results/260715_0022__vs__260715_0022', '--sim_eps', '350', '--trial_seq', '/home/tom/repos/agent_training/trial_sequences/trials_100000_seed17.json', '--seed', '17']
=== Finished: 260715_0022__vs__260715_0022 -> /home/tom/repos/agent_training/tournament/results/260715_0022__vs__260715_0022/mlagents_stdout.log ===

=== [2/64] Matchup: 260715_0022 (P1) vs 260715_0005 (P2) ===
['mlagents-learn', '/home/tom/repo

In [ ]:
# Summary: which matchups produced a log, which failed.
ok = {k: v for k, v in logs.items() if v is not None}
failed = [k for k, v in logs.items() if v is None]
print(f"{len(ok)}/{len(logs)} matchups OK")
for k in failed:
    print("  FAILED:", k)